# Homework 08 - PyTorch Bridge

目标：把 micrograd 的动作逐个翻译到 PyTorch。PyTorch 需要装在本地 venv，不要装系统 Python。

In [5]:
from pathlib import Path
import sys, math, random


def find_repo_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'micrograd' / 'engine.py').exists():
            return q
    return p

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def near(a, b, eps=1e-6):
    return abs(a - b) < eps


def todo_guard(values, message='请先填写 TODO，再运行检查。'):
    if any(v is None for v in values):
        print(message)
        return False
    return True


def check_or_todo(condition, message):
    if not condition:
        print(message)
        return False
    return True

from course.checks import qa_check

## 完整例子 - micrograd 一步训练

In [6]:
from micrograd.engine import Value

w = Value(2.0)
x = Value(3.0)
y = Value(7.0)
pred = w * x
loss = (pred - y) ** 2
loss.backward()
print('micrograd pred/loss/grad:', pred.data, loss.data, w.grad)
w.data -= 0.1 * w.grad
print('updated w:', w.data)

micrograd pred/loss/grad: 6.0 1.0 -6.0
updated w: 2.6


## 作业 1 - micrograd 和 PyTorch 跑同一个例子


In [7]:
# 填写说明：
# - 这题拆成三步：先 micrograd，再 PyTorch，最后对比。
# - student_micrograd_one_step 返回顺序：grad、updated_w。
# - student_torch_one_step_homework 返回顺序：grad、updated_w。
# - student_compare_micrograd_torch 调用前两个函数，return micrograd_result, torch_result。
# - 运行本 cell，看 qa_check 的提示。

from micrograd.engine import Value
try:
    import torch
except Exception:
    torch = None


def student_micrograd_one_step():
    # TODO 1: w=2, x=3, y=7, loss=(w*x-y)^2
    # TODO 2: backward 后记录 w.grad
    # TODO 3: 用 lr=0.1 更新 w.data
    # TODO 4: return grad, updated_w
    w = Value(2.0)
    x = Value(3.0)
    y = Value(7.0)
    pred = w * x
    loss = (pred - y) ** 2
    loss.backward()
    print('micrograd pred/loss/grad:', pred.data, loss.data, w.grad)

    lr = 0.1
    w.data -= lr * w.grad
    return w.grad, w.data;


def student_torch_one_step_homework():
    if torch is None:
        print('torch 未安装。先运行: .venv/bin/python -m pip install -r course/requirements-course.txt')
        return None
    # TODO: 参考 student_micrograd_one_step，用 PyTorch 做同一件事。
    w_t = torch.tensor(2.0, requires_grad=True)
    x_t = torch.tensor(3.0)
    y_t = torch.tensor(7.0)

    pred_t = w_t * x_t
    loss_t = (pred_t - y_t) ** 2
    
    print('pred:', pred_t.item())
    print('loss:', loss_t.item())
    
    loss_t.backward()
    print('w.grad:', w_t.grad.item())

    lr = 0.1
    with torch.no_grad():
        w_t += -lr * w_t.grad
    
    return w_t.grad.item(), w_t.item();



def student_compare_micrograd_torch():
    # TODO: 调用前两个函数，return micrograd_result, torch_result
    return micrograd_result(), 


qa_check('qa_check_08_mapping', globals(), student_compare_micrograd_torch)


请先完成 TODO：函数还返回了 None。
请返回一个 dict，这样每个中间结果都能被检查。


<details><summary>Show / Hide 本题提示</summary>

- 先写 micrograd 版本，再照着把 `Value` 换成 `torch.tensor(..., requires_grad=True)`。
- 两边都应该得到同一个梯度：-6。
- 更新公式都还是 `w_new = w - lr * grad`。

</details>


<details><summary>Show / Hide 本题答案</summary>

```python
def student_micrograd_one_step():
    w = Value(2.0)
    x = Value(3.0)
    y = Value(7.0)
    loss = (w * x - y) ** 2
    loss.backward()
    grad = w.grad
    w.data -= 0.1 * w.grad
    return grad, w.data


def student_torch_one_step_homework():
    w = torch.tensor(2.0, requires_grad=True)
    x = torch.tensor(3.0)
    y = torch.tensor(7.0)
    loss = (w * x - y) ** 2
    loss.backward()
    grad = w.grad.item()
    with torch.no_grad():
        w -= 0.1 * w.grad
    return grad, w.item()


def student_compare_micrograd_torch():
    return student_micrograd_one_step(), student_torch_one_step_homework()
```

</details>


## Modify - 同一件事，换学习率

micrograd 和 PyTorch 的更新本质一样：`new = old - lr * grad`。

In [ ]:
# 填写说明：
# - 填一个数字：lr=0.01 时的新 w。
# - 填完后运行本 cell，看 qa_check 的提示。

old_w = 2.0
grad = -6.0
# TODO: lr=0.01 时，新 w 是多少？
student_w_after_lr_001 = None



qa_check('qa_check_08_modify', globals())

<details><summary>Show / Hide 本题提示</summary>

- 把 micrograd 词汇翻译过去：`Value -> Tensor(requires_grad=True)`，`backward -> loss.backward()`，更新由 optimizer 接手。

</details>

<details><summary>Show / Hide 本题答案</summary>


参考答案（先自己做，卡住再展开）：

- `student_w_after_lr_001` 约等于 `2.06`。

</details>

## 作业 2 - PyTorch 一步训练，可选实际运行

如果当前 kernel 没有 torch，这个 cell 会跳过运行，但概念题必须完成。

In [ ]:
try:
    import torch
except Exception as exc:
    torch = None
    print('torch 未安装。请用本地 venv：python3 -m venv .venv && .venv/bin/python -m pip install -r course/requirements-course.txt')

if torch is not None:
    w_t = torch.tensor(2.0, requires_grad=True)
    x_t = torch.tensor(3.0)
    y_t = torch.tensor(7.0)
    pred_t = w_t * x_t
    loss_t = (pred_t - y_t) ** 2
    loss_t.backward()
    print('torch pred/loss/grad:', pred_t.item(), loss_t.item(), w_t.grad.item())
    with torch.no_grad():
        w_t -= 0.1 * w_t.grad
    w_t.grad.zero_()
    print('updated w:', w_t.item(), 'grad after zero:', w_t.grad.item())
else:
    print('跳过 torch 运行；先把上面的映射写对。')

## Debug Lab - PyTorch 最常见两个坑

In [ ]:
# 填写说明：
# - 这题拆成两个小函数：一个练 no_grad 更新，一个练 Tensor 批量计算。
# - student_torch_update_demo 返回 updated_w。
# - student_tensor_batch_demo 返回顺序：batch_shape、batch_times_two。
# - student_torch_update_and_batch_demo 调用前两个函数。
# - 运行本 cell，看 qa_check 的提示。

def student_torch_update_demo():
    if torch is None:
        print('torch 未安装。先运行: .venv/bin/python -m pip install -r course/requirements-course.txt')
        return None
    # TODO: w=2, loss=(w*3-7)^2，backward 后用 no_grad 更新 w，return updated_w。
    pass


def student_tensor_batch_demo():
    if torch is None:
        print('torch 未安装。先运行: .venv/bin/python -m pip install -r course/requirements-course.txt')
        return None
    # TODO: 创建 batch=[1,2,3]，计算 batch*2，return batch_shape, batch_times_two。
    pass


def student_torch_update_and_batch_demo():
    # TODO: 调用前两个函数，return updated_w, batch_shape, batch_times_two
    pass


qa_check('qa_check_08_debug', globals())


<details><summary>Show / Hide 本题提示</summary>

- `torch.no_grad()` 的重点：手动更新参数这一步不需要再被 autograd 记录。
- `Value` 是标量；Tensor 可以一次装一组数，所以 `batch * 2` 会得到一组结果。

</details>


<details><summary>Show / Hide 本题答案</summary>

```python
def student_torch_update_demo():
    w = torch.tensor(2.0, requires_grad=True)
    loss = (w * 3 - 7) ** 2
    loss.backward()
    with torch.no_grad():
        w -= 0.1 * w.grad
    return w.item()


def student_tensor_batch_demo():
    batch = torch.tensor([1.0, 2.0, 3.0])
    doubled = batch * 2
    return tuple(batch.shape), doubled.tolist()


def student_torch_update_and_batch_demo():
    updated_w = student_torch_update_demo()
    batch_shape, batch_times_two = student_tensor_batch_demo()
    return updated_w, batch_shape, batch_times_two
```

</details>


## 提交清单

- [ ] micrograd 和 PyTorch 的同一个小例子对齐通过
- [ ] 如果环境有 torch，实际跑过一步训练
- [ ] Debug Lab 通过
- [ ] 我能说出 PyTorch 封装了 micrograd 的哪些手写动作


<details><summary>Show / Hide 提示</summary>

先找完整例子的形状，再只改一个地方。填 TODO 前，先用小数字在纸上算一遍；测试失败时先判断错在数学、Python、计算图还是训练循环。

</details>

<details><summary>Show / Hide 答案</summary>

答案不要一开始看。正确答案应该能由完整例子和 Modify 题一步推出；如果你需要直接看答案，说明前一个台阶还没踩稳。

</details>